# CORES + GETTEL Explorer — Build Pipeline

Run the cells below **in order, top to bottom**. Each block is one Colab cell:

1. **Data Ingestion & Entity Resolution** — upload your 5 source files (Gettel `.xlsx` + 4 CORES CSVs) when prompted.
2. **Network Mind Map (HTML)**
3. **Transaction Dashboard (HTML)**
4. **Assemble Final HTML** — combines 2+3 into `cores_gettel_explorer.html`
5. **Download** — pulls the finished file to your computer

No need to install anything — Colab already has `pandas` and `openpyxl`. Just run each cell (Shift+Enter) and wait for it to finish before running the next.


In [ ]:
# @title 1 — DATA INGESTION & ENTITY RESOLUTION
# Inputs: Gettel .xlsx + 4 CORES CSVs
# Outputs: graph_data.json, dashboard_data.json, gettel_scrape_list.csv, gettel_transactions_parsed.csv

import pandas as pd
import json
import re
import io
import os
from datetime import datetime
from collections import defaultdict
import openpyxl
from google.colab import files

# ── 1A: UPLOAD FILES ─────────────────────────────────────────────────────────

print("=" * 60)
print("UPLOAD REQUIRED — 5 files total")
print("=" * 60)
print()
print("Please upload the following files when prompted:")
print()
print("  1. Gettel Excel (the transaction database):")
print("     → 2026_02_19_Gettel_Sales_2018-2026_-with_links.xlsx")
print()
print("  2. CORES Phase 1 companies CSV:")
print("     → cores_companies_[timestamp].csv")
print()
print("  3. CORES Phase 1 directors CSV:")
print("     → cores_directors_[timestamp].csv")
print()
print("  4. CORES Phase 2 (deep) companies CSV:")
print("     → cores_companies_deep_[timestamp].csv")
print()
print("  5. CORES Phase 2 (deep) directors CSV:")
print("     → cores_directors_deep_[timestamp].csv")
print()
print("Select all 5 at once in the file picker (Ctrl+click / Cmd+click).")
print("=" * 60)

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No files uploaded. Re-run this cell and upload all 5 files.")

uploaded_names = list(uploaded.keys())
print(f"\nUploaded {len(uploaded_names)} file(s):")
for n in uploaded_names:
    print(f"  {n}")

# ── Match uploaded files by pattern ──────────────────────────────────────────

def find_file(pattern_keywords, uploaded_names, require=True):
    for name in uploaded_names:
        lower = name.lower()
        if all(kw in lower for kw in pattern_keywords):
            return name
    if require:
        raise FileNotFoundError(
            f"Could not find a file matching keywords {pattern_keywords} "
            f"among uploaded files: {uploaded_names}\n"
            f"Please re-run the cell and upload all 5 required files."
        )
    return None

gettel_file = find_file(['.xlsx'], uploaded_names)
p1_cos_file  = find_file(['companies', 'deep'], uploaded_names, require=False)
p1_dirs_file = find_file(['directors', 'deep'], uploaded_names, require=False)
# Phase 1 files must NOT contain 'deep'
p1_cos_file  = next((n for n in uploaded_names if 'companies' in n.lower() and 'deep' not in n.lower()), None)
p1_dirs_file = next((n for n in uploaded_names if 'directors' in n.lower() and 'deep' not in n.lower()), None)
p2_cos_file  = next((n for n in uploaded_names if 'companies' in n.lower() and 'deep' in n.lower()), None)
p2_dirs_file = next((n for n in uploaded_names if 'directors' in n.lower() and 'deep' in n.lower()), None)

missing = []
if not gettel_file:  missing.append("Gettel .xlsx")
if not p1_cos_file:  missing.append("cores_companies (Phase 1, no 'deep' in name)")
if not p1_dirs_file: missing.append("cores_directors (Phase 1, no 'deep' in name)")
if not p2_cos_file:  missing.append("cores_companies_deep (Phase 2)")
if not p2_dirs_file: missing.append("cores_directors_deep (Phase 2)")

if missing:
    raise FileNotFoundError(
        f"\nMissing files — could not identify:\n" +
        "\n".join(f"  - {m}" for m in missing) +
        f"\n\nUploaded: {uploaded_names}\n"
        f"Re-run the cell and upload all 5 files with the correct names."
    )

print(f"\nFile mapping:")
print(f"  Gettel      → {gettel_file}")
print(f"  P1 Companies→ {p1_cos_file}")
print(f"  P1 Directors→ {p1_dirs_file}")
print(f"  P2 Companies→ {p2_cos_file}")
print(f"  P2 Directors→ {p2_dirs_file}")

# ── 1A: LOAD SOURCES ─────────────────────────────────────────────────────────

print("\nLoading Gettel...")
wb = openpyxl.load_workbook(io.BytesIO(uploaded[gettel_file]), data_only=True)

# Auto-detect sheet name — falls back to first sheet if Sheet5 not found
sheet_name = "Sheet5" if "Sheet5" in wb.sheetnames else wb.sheetnames[0]
if sheet_name != "Sheet5":
    print(f"  Note: 'Sheet5' not found — using sheet '{sheet_name}'")
ws = wb[sheet_name]
rows = list(ws.iter_rows(values_only=True))
gettel_headers = rows[0]
gettel_raw = pd.DataFrame(rows[1:], columns=gettel_headers)
print(f"  Gettel: {len(gettel_raw)} rows, {len(gettel_raw.columns)} cols")

print("Loading CORES...")
p1_cos  = pd.read_csv(io.BytesIO(uploaded[p1_cos_file]),  dtype=str)
p1_dirs = pd.read_csv(io.BytesIO(uploaded[p1_dirs_file]), dtype=str)
p2_cos  = pd.read_csv(io.BytesIO(uploaded[p2_cos_file]),  dtype=str)
p2_dirs = pd.read_csv(io.BytesIO(uploaded[p2_dirs_file]), dtype=str)

p1_cos["depth"]      = "1"
p1_cos["parent_CAN"] = ""
p1_dirs["depth"]     = "1"
p1_dirs["parent_CAN"]= ""

all_companies = pd.concat([p1_cos, p2_cos], ignore_index=True)
all_directors = pd.concat([p1_dirs, p2_dirs], ignore_index=True)
print(f"  CORES (raw): {len(all_companies)} companies, {len(all_directors)} director records")

# ── 1A: DEDUPE ───────────────────────────────────────────────────────────────
# The scraper re-visits companies across runs/phases and re-appends rows, so the
# same CAN (and the same director) can show up several times. Most repeats are
# byte-for-byte identical, but a few are the same company/director scraped at
# different points in time (a new annual return filed, a status change, a
# parent_CAN only found once the deep crawl ran). We keep one row per key,
# preferring the most complete / most recently-filed version rather than just
# the first one seen, so we don't silently regress a status or drop a
# parent_CAN link that a later scrape picked up.

def _completeness(df, ignore_cols=()):
    """Count of non-null, non-empty-string fields per row (higher = richer record)."""
    scored = df.drop(columns=[c for c in ignore_cols if c in df.columns])
    return scored.apply(lambda col: col.notna() & (col.astype(str).str.strip() != ""), axis=0).sum(axis=1)

def dedupe_companies(df):
    df = df.copy()
    ar_year = pd.to_numeric(df.get("Last_AR_Year"), errors="coerce").fillna(-1)
    completeness = _completeness(df, ignore_cols=["CAN"])
    df = (df.assign(_ar_year=ar_year, _completeness=completeness)
            .sort_values(["_ar_year", "_completeness"], ascending=[False, False])
            .drop(columns=["_ar_year", "_completeness"]))
    before = len(df)
    df = df.drop_duplicates(subset=["CAN"], keep="first").reset_index(drop=True)
    print(f"  Deduped companies: {before} -> {len(df)} ({before - len(df)} duplicate rows removed)")
    return df

def dedupe_directors(df):
    # Appointment_Date is part of the key: a person can resign and be
    # reappointed later, which is two legitimate records, not a duplicate.
    key_cols = ["CAN", "Last_Name", "First_Name", "Type", "Appointment_Date"]
    df = df.copy()
    completeness = _completeness(df, ignore_cols=key_cols)
    df = (df.assign(_completeness=completeness)
            .sort_values("_completeness", ascending=False)
            .drop(columns=["_completeness"]))
    before = len(df)
    df = df.drop_duplicates(subset=key_cols, keep="first").reset_index(drop=True)
    print(f"  Deduped directors: {before} -> {len(df)} ({before - len(df)} duplicate rows removed)")
    return df

all_companies = dedupe_companies(all_companies)
all_directors = dedupe_directors(all_directors)
print(f"  CORES (deduped): {len(all_companies)} companies, {len(all_directors)} director records")

# ── 1A: NORMALIZE ────────────────────────────────────────────────────────────

CORP_KEYWORDS = [
    r'\bLTD\.?\b', r'\bINC\.?\b', r'\bCORP\.?\b', r'\bCORPORATION\b',
    r'\bLIMITED\b', r'\bHOLDINGS\b', r'\bENTERPRISES\b', r'\bPROPERTIES\b',
    r'\bREALTY\b', r'\bDEVELOPMENTS?\b', r'\bINVESTMENTS?\b', r'\bGROUP\b',
    r'\bPARTNERS?\b', r'\bASSOCIATES?\b', r'\bCAPITAL\b', r'\bMANAGEMENT\b',
    r'\bRESIDENCES?\b', r'\bVENTURES?\b', r'\bINDUSTRIES\b', r'\bSERVICES\b',
    r'\bL\.L\.C\b', r'\bLLP\b', r'\bLP\b', r'\bTRUST\b',
]
CORP_RE = re.compile('|'.join(CORP_KEYWORDS), re.I)

GOVT_KEYWORDS = [
    'CITY OF', 'TOWN OF', 'COUNTY OF', 'MUNICIPALITY OF', 'PROVINCE OF',
    'ALBERTA INFRASTRUCTURE', 'ALBERTA TRANSPORTATION', 'GOVERNMENT OF',
    'FEDERAL', 'CROWN', 'SCHOOL BOARD', 'SCHOOL DIVISION',
]

ET_AL_RE    = re.compile(r'\s*(Et,?\s*al\.?|et\s*al\.?)\s*$', re.I)
ROLE_RE     = re.compile(r'^(directors?|shareholders?|director/shareholder|dshareholder)[;:]?\s*(.*?)$', re.I)
NUMBERED_RE = re.compile(r'^\d{6,}')

def normalize(name):
    if not isinstance(name, str):
        return ""
    n = name.upper().strip()
    n = ET_AL_RE.sub('', n)
    n = re.sub(r'\s+', ' ', n)
    n = n.rstrip('.,;').strip()
    return n

def detect_entity_type(name):
    up = name.upper()
    for kw in GOVT_KEYWORDS:
        if kw in up:
            return "government"
    if NUMBERED_RE.match(name.strip()):
        return "company"
    if CORP_RE.search(name):
        return "company"
    return "individual"

def detect_province(name):
    up = name.upper()
    if re.search(r'\bALBERTA\b|^\d+\s+ALBERTA', up):
        return "AB"
    if re.search(r'\bB\.C\.\b|\bBC\b|\bBRITISH COLUMBIA\b', up):
        return "BC"
    if re.search(r'\bONTARIO\b|\bONT\.\b', up):
        return "ON"
    if re.search(r'\bSASKATCHEWAN\b|\bMANITOBA\b|\bQUEBEC\b|\bNOVA SCOTIA\b', up):
        return "OTHER"
    return "unknown"

cores_can_lookup = {normalize(row["Legal_Name"]): str(row["CAN"]) for _, row in all_companies.iterrows()}

# ── 1B: PARSE GETTEL VENDOR / PURCHASER ──────────────────────────────────────

def parse_party(raw_text):
    if not isinstance(raw_text, str) or not raw_text.strip():
        return {"entity":"", "entity_type":"", "person":"", "role":"", "address":"", "is_et_al":False}

    lines = [l.strip() for l in raw_text.split('\n') if l.strip()]
    if not lines:
        return {"entity":"", "entity_type":"", "person":"", "role":"", "address":"", "is_et_al":False}

    raw_entity = lines[0]
    is_et_al   = bool(ET_AL_RE.search(raw_entity))
    entity     = ET_AL_RE.sub('', raw_entity).strip()
    entity_type= detect_entity_type(entity)

    person  = ""
    role    = ""
    addr_lines = []

    if len(lines) > 1:
        idx = 1
        m = ROLE_RE.match(lines[idx]) if idx < len(lines) else None
        if m:
            raw_role = m.group(1).lower()
            role = "Shareholder" if "share" in raw_role else "Director"
            inline_name = m.group(2).strip()
            if inline_name:
                person = inline_name
                idx += 1
            else:
                idx += 1
                if idx < len(lines) and not is_address_line(lines[idx]):
                    person = lines[idx]
                    idx += 1
            addr_lines = lines[idx:]
        else:
            if entity_type == "individual":
                addr_lines = lines[1:]
            else:
                if idx < len(lines) and not is_address_line(lines[idx]):
                    person = lines[idx]
                    idx += 1
                addr_lines = lines[idx:]

    address = ", ".join(addr_lines)
    return {"entity": entity, "entity_type": entity_type, "person": person,
            "role": role, "address": address, "is_et_al": is_et_al}

def is_address_line(line):
    if re.match(r'^\d{1,6}\s', line):
        return True
    if re.match(r'^(Box|Suite|Unit|RR|PO|P\.O\.)', line, re.I):
        return True
    if re.search(r'\b(AB|BC|ON|SK|MB|QC)\b\s+[A-Z]\d[A-Z]', line):
        return True
    return False

gettel_cols = list(gettel_raw.columns)

def get_col(row, idx):
    try:
        return row.iloc[idx]
    except:
        return None

print("Parsing Gettel vendor/purchaser fields...")
records = []
for i, row in gettel_raw.iterrows():
    vp = parse_party(get_col(row, 11))
    pp = parse_party(get_col(row, 12))

    sale_price = get_col(row, 19)
    try:
        sale_price = float(sale_price) if sale_price else None
    except:
        sale_price = None

    sale_date = get_col(row, 20)
    sale_date_str = sale_date.strftime('%Y-%m-%d') if hasattr(sale_date, 'strftime') else str(sale_date or '')

    sale_year = get_col(row, 21)
    try:
        sale_year = int(sale_year) if sale_year else None
    except:
        sale_year = None

    unit_price = get_col(row, 22)
    try:
        unit_price = float(unit_price) if unit_price else None
    except:
        unit_price = None

    site_area = get_col(row, 15)
    try:
        site_area = float(site_area) if site_area else None
    except:
        site_area = None

    bldg_area = get_col(row, 17)
    try:
        bldg_area = float(bldg_area) if bldg_area else None
    except:
        bldg_area = None

    year_built = get_col(row, 25)
    try:
        year_built = int(year_built) if year_built else None
    except:
        year_built = None

    ownership_type = str(get_col(row, 6) or '').strip().title()

    records.append({
        "txn_id":             i,
        "prop_id":            str(get_col(row, 3) or ''),
        "property_class":     str(get_col(row, 4) or ''),
        "property_type":      str(get_col(row, 5) or ''),
        "ownership_type":     ownership_type,
        "description":        str(get_col(row, 7) or ''),
        "land_use":           str(get_col(row, 8) or ''),
        "address":            str(get_col(row, 9) or ''),
        "city":               str(get_col(row, 10) or ''),
        "legal_description":  str(get_col(row, 13) or ''),
        "subdivision":        str(get_col(row, 14) or ''),
        "site_area":          site_area,
        "site_units":         str(get_col(row, 16) or ''),
        "bldg_area":          bldg_area,
        "bldg_units":         str(get_col(row, 18) or ''),
        "sale_price":         sale_price,
        "sale_date":          sale_date_str,
        "sale_year":          sale_year,
        "unit_price":         unit_price,
        "unit_measure":       str(get_col(row, 23) or ''),
        "year_built":         year_built,
        "vendor_raw":         str(get_col(row, 11) or ''),
        "vendor_entity":      vp["entity"],
        "vendor_entity_type": vp["entity_type"],
        "vendor_person":      vp["person"],
        "vendor_role":        vp["role"],
        "vendor_address":     vp["address"],
        "vendor_is_et_al":    vp["is_et_al"],
        "purchaser_raw":      str(get_col(row, 12) or ''),
        "purchaser_entity":   pp["entity"],
        "purchaser_entity_type": pp["entity_type"],
        "purchaser_person":   pp["person"],
        "purchaser_role":     pp["role"],
        "purchaser_address":  pp["address"],
        "purchaser_is_et_al": pp["is_et_al"],
    })

gettel_txns = pd.DataFrame(records)

# Parse stats
print(f"\nParse summary:")
print(f"  Vendor entity types: {gettel_txns['vendor_entity_type'].value_counts().to_dict()}")
print(f"  Purchaser entity types: {gettel_txns['purchaser_entity_type'].value_counts().to_dict()}")
print(f"  Vendor roles: {gettel_txns['vendor_role'].value_counts().to_dict()}")
print(f"  Vendor et al: {gettel_txns['vendor_is_et_al'].sum()}")
print(f"  Purchaser et al: {gettel_txns['purchaser_is_et_al'].sum()}")

# ── 1C: ENTITY MATCHING ───────────────────────────────────────────────────────

print("\nMatching Gettel entities to CORES...")

def match_company(name):
    norm = normalize(name)
    if norm in cores_can_lookup:
        return cores_can_lookup[norm], "exact"
    # Try stripping common suffix variants
    alt = re.sub(r'\bLTD$|LIMITED$', 'LTD.', norm)
    if alt in cores_can_lookup:
        return cores_can_lookup[alt], "normalized"
    alt2 = norm.rstrip('.')
    if alt2 in cores_can_lookup:
        return cores_can_lookup[alt2], "normalized"
    return "", "unmatched"

vendor_cans    = {}
purchaser_cans = {}

for ent in gettel_txns["vendor_entity"].unique():
    if ent and detect_entity_type(ent) == "company":
        can, conf = match_company(ent)
        vendor_cans[ent] = can

for ent in gettel_txns["purchaser_entity"].unique():
    if ent and detect_entity_type(ent) == "company":
        can, conf = match_company(ent)
        purchaser_cans[ent] = can

gettel_txns["vendor_can"]    = gettel_txns["vendor_entity"].map(lambda e: vendor_cans.get(e, ""))
gettel_txns["purchaser_can"] = gettel_txns["purchaser_entity"].map(lambda e: purchaser_cans.get(e, ""))

matched_v = (gettel_txns["vendor_can"] != "").sum()
matched_p = (gettel_txns["purchaser_can"] != "").sum()
print(f"  Vendor matches: {matched_v} / {len(gettel_txns)}")
print(f"  Purchaser matches: {matched_p} / {len(gettel_txns)}")

# Person matching
cores_person_lookup = defaultdict(list)
for _, row in all_directors.iterrows():
    if str(row.get("Individual_or_Corp","")).strip() == "Individual":
        last  = str(row.get("Last_Name","")).strip().upper()
        first = str(row.get("First_Name","")).strip().upper()
        if last:
            cores_person_lookup[(last, first)].append({
                "can":          str(row.get("CAN","")),
                "company_name": str(row.get("Company_Name","")),
                "role":         str(row.get("Type","")),
                "city":         str(row.get("City","")),
                "province":     str(row.get("Province","")),
            })

def split_person_name(full):
    parts = full.strip().split()
    if len(parts) >= 2:
        return parts[-1].upper(), " ".join(parts[:-1]).upper()
    return full.upper(), ""

person_edges = []
for _, row in gettel_txns.iterrows():
    for side in [("vendor","vendor_person","vendor_can"), ("purchaser","purchaser_person","purchaser_can")]:
        side_name, person_col, can_col = side
        person = str(row.get(person_col,"")).strip()
        company_can = str(row.get(can_col,"")).strip()
        if not person:
            continue
        last, first = split_person_name(person)
        matches = cores_person_lookup.get((last, first), [])
        if not matches:
            matches = cores_person_lookup.get((last, ""), [])
        for m in matches:
            confidence = "high_direct" if m["can"] == company_can and company_can else "low_name"
            person_edges.append({
                "person_name":   person,
                "person_key":    f"{last},{first}",
                "company_can":   company_can or m["can"],
                "txn_id":        int(row["txn_id"]),
                "confidence":    confidence,
                "role":          side_name,
            })

# Medium confidence: CORES directors of transacting companies not named in Gettel
transacting_cans = set(gettel_txns["vendor_can"].dropna()) | set(gettel_txns["purchaser_can"].dropna())
transacting_cans.discard("")
for _, row in all_directors.iterrows():
    if str(row.get("Individual_or_Corp","")).strip() != "Individual":
        continue
    can = str(row.get("CAN","")).strip()
    if can not in transacting_cans:
        continue
    last  = str(row.get("Last_Name","")).strip().upper()
    first = str(row.get("First_Name","")).strip().upper()
    person_key = f"{last},{first}"
    if not any(e["person_key"] == person_key and e["company_can"] == can for e in person_edges):
        person_edges.append({
            "person_name": f"{first} {last}".strip(),
            "person_key":  person_key,
            "company_can": can,
            "txn_id":      None,
            "confidence":  "medium_cores",
            "role":        str(row.get("Type","")).lower(),
        })

print(f"  Person edges built: {len(person_edges)}")

# ── 1D: BUILD GRAPH DATA ──────────────────────────────────────────────────────

print("\nBuilding graph_data.json...")

def safe_str(v):
    return "" if pd.isna(v) or v is None else str(v)

# Index transactions by CAN
txns_by_can = defaultdict(list)
for _, row in gettel_txns.iterrows():
    entry = {
        "txn_id":         int(row["txn_id"]),
        "sale_price":     row["sale_price"],
        "sale_date":      row["sale_date"],
        "property_class": row["property_class"],
        "property_type":  row["property_type"],
        "description":    row["description"],
        "address":        row["address"],
        "city":           row["city"],
        "site_area":      row["site_area"],
        "site_units":     row["site_units"],
        "bldg_area":      row["bldg_area"],
        "unit_price":     row["unit_price"],
        "year_built":     row["year_built"],
    }
    if row["vendor_can"]:
        txns_by_can[row["vendor_can"]].append({**entry, "role":"vendor",
            "counterparty": row["purchaser_entity"],
            "counterparty_person": row["purchaser_person"]})
    if row["purchaser_can"]:
        txns_by_can[row["purchaser_can"]].append({**entry, "role":"purchaser",
            "counterparty": row["vendor_entity"],
            "counterparty_person": row["vendor_person"]})

# Build directors index per company
dirs_by_can = defaultdict(list)
for _, row in all_directors.iterrows():
    can = safe_str(row.get("CAN",""))
    dirs_by_can[can].append({
        "id":           f"{safe_str(row.get('Last_Name',''))}_{safe_str(row.get('First_Name',''))}_{can}",
        "label":        f"{safe_str(row.get('Last_Name',''))}, {safe_str(row.get('First_Name',''))}".strip(', '),
        "node_type":    safe_str(row.get("Individual_or_Corp","individual")).lower(),
        "role":         safe_str(row.get("Type","")),
        "pct_shares":   safe_str(row.get("Percent_Voting_Shares","")),
        "status":       safe_str(row.get("Status","")),
        "appointment":  safe_str(row.get("Appointment_Date","")),
        "cessation":    safe_str(row.get("Cessation_Date","")),
        "address":      ", ".join(filter(None, [
                            safe_str(row.get("Street","")),
                            safe_str(row.get("City","")),
                            safe_str(row.get("Province","")),
                            safe_str(row.get("Postal",""))])),
        "corp_can":     safe_str(row.get("Director_Corp_CAN","")),
        "source":       "cores",
    })

company_info = {}

# CORES companies
for _, row in all_companies.iterrows():
    can = safe_str(row.get("CAN",""))
    txns = txns_by_can.get(can, [])
    company_info[can] = {
        "can":               can,
        "label":             safe_str(row.get("Legal_Name","")),
        "status":            safe_str(row.get("Status","")),
        "depth":             safe_str(row.get("depth","1")),
        "parent_can":        safe_str(row.get("parent_CAN","")),
        "city":              safe_str(row.get("Reg_City","")),
        "le_type":           safe_str(row.get("LE_Type","")),
        "corp_type":         safe_str(row.get("Corp_Type","")),
        "registration_date": safe_str(row.get("Registration_Date","")),
        "address":           ", ".join(filter(None, [
                                safe_str(row.get("Reg_Street","")),
                                safe_str(row.get("Reg_City","")),
                                safe_str(row.get("Reg_Province","")),
                                safe_str(row.get("Reg_Postal",""))])),
        "email":             safe_str(row.get("Email","")),
        "agent":             safe_str(row.get("Agent_For_Service","")),
        "last_ar_year":      safe_str(row.get("Last_AR_Year","")),
        "last_ar_filed":     safe_str(row.get("Last_AR_Filed","")),
        "all_annual_returns":safe_str(row.get("All_Annual_Returns","")),
        "source":            "cores" if not txns else "cores+gettel",
        "children":          dirs_by_can.get(can, []),
        "transactions":      txns,
    }

# Gettel-only companies (not in CORES)
all_cores_cans = set(company_info.keys())
gettel_only_entities = {}
for _, row in gettel_txns.iterrows():
    for side, can_col in [("vendor","vendor_can"),("purchaser","purchaser_can")]:
        ent  = row[f"{side}_entity"]
        can  = row[f"{side}_can"]
        etype= row[f"{side}_entity_type"]
        if ent and etype == "company" and (not can or can not in all_cores_cans):
            key = normalize(ent)
            if key not in gettel_only_entities:
                gettel_only_entities[key] = {"label": ent, "txns": [], "persons": {}}
            t_entry = {
                "txn_id":         int(row["txn_id"]),
                "sale_price":     row["sale_price"],
                "sale_date":      row["sale_date"],
                "property_class": row["property_class"],
                "description":    row["description"],
                "address":        row["address"],
                "city":           row["city"],
            }
            t_entry["role"] = side
            t_entry["counterparty"] = row["purchaser_entity" if side=="vendor" else "vendor_entity"]
            gettel_only_entities[key]["txns"].append(t_entry)
            person = row[f"{side}_person"]
            role   = row[f"{side}_role"]
            if person:
                gettel_only_entities[key]["persons"][person] = role

for key, data in gettel_only_entities.items():
    fake_can = f"GETTEL_{key[:30]}"
    children = [{"id": f"p_{normalize(p)}", "label": p, "node_type": "individual",
                 "role": r, "pct_shares":"","status":"","appointment":"",
                 "cessation":"","address":"","corp_can":"","source":"gettel"}
                for p, r in data["persons"].items()]
    company_info[fake_can] = {
        "can": fake_can, "label": data["label"], "status": "", "depth": "1",
        "parent_can": "", "city": "", "le_type": "", "corp_type": "",
        "registration_date": "", "address": "", "email": "", "agent": "",
        "last_ar_year": "", "last_ar_filed": "", "all_annual_returns": "",
        "source": "gettel_only", "children": children, "transactions": data["txns"],
    }

# Person index
person_index = defaultdict(lambda: {"label":"","companies":[],"transactions":[]})
for e in person_edges:
    key = e["person_key"]
    person_index[key]["label"] = e["person_name"]
    if e["company_can"] and e["company_can"] not in person_index[key]["companies"]:
        person_index[key]["companies"].append(e["company_can"])
    if e["txn_id"] is not None:
        person_index[key]["transactions"].append({
            "txn_id":     e["txn_id"],
            "confidence": e["confidence"],
            "company_can":e["company_can"],
            "role":       e["role"],
        })

company_list = [
    {"can": v["can"], "label": v["label"], "status": v["status"],
     "depth": v["depth"], "city": v["city"],
     "child_count": len(v["children"]), "txn_count": len(v["transactions"]),
     "source": v["source"]}
    for v in company_info.values()
]

graph_data = {
    "company_info": company_info,
    "company_list": company_list,
    "person_index": dict(person_index),
    "meta": {
        "total_companies": len(company_info),
        "total_persons":   len(person_index),
        "total_transactions": len(gettel_txns),
        "generated": datetime.now().strftime("%Y-%m-%d"),
    }
}

with open("graph_data.json","w",encoding="utf-8") as f:
    json.dump(graph_data, f, default=str)
print(f"  graph_data.json: {len(company_info)} companies, {len(person_index)} persons")

# ── 1E: BUILD DASHBOARD DATA ──────────────────────────────────────────────────

print("Building dashboard_data.json...")

txn_records = []
for _, row in gettel_txns.iterrows():
    txn_records.append({
        "txn_id":           int(row["txn_id"]),
        "prop_id":          row["prop_id"],
        "property_class":   row["property_class"],
        "property_type":    row["property_type"],
        "ownership_type":   row["ownership_type"],
        "description":      row["description"],
        "land_use":         row["land_use"],
        "address":          row["address"],
        "city":             row["city"],
        "subdivision":      row["subdivision"],
        "site_area":        row["site_area"],
        "site_units":       row["site_units"],
        "bldg_area":        row["bldg_area"],
        "bldg_units":       row["bldg_units"],
        "sale_price":       row["sale_price"],
        "sale_date":        row["sale_date"],
        "sale_year":        row["sale_year"],
        "unit_price":       row["unit_price"],
        "year_built":       row["year_built"],
        "vendor_entity":    row["vendor_entity"],
        "vendor_person":    row["vendor_person"],
        "vendor_role":      row["vendor_role"],
        "vendor_type":      row["vendor_entity_type"],
        "purchaser_entity": row["purchaser_entity"],
        "purchaser_person": row["purchaser_person"],
        "purchaser_role":   row["purchaser_role"],
        "purchaser_type":   row["purchaser_entity_type"],
        "vendor_can":       row["vendor_can"],
        "purchaser_can":    row["purchaser_can"],
        "vendor_is_et_al":  row["vendor_is_et_al"],
        "purchaser_is_et_al": row["purchaser_is_et_al"],
    })

prices = [r["sale_price"] for r in txn_records if r["sale_price"]]
dashboard_data = {
    "transactions": txn_records,
    "filter_options": {
        "property_classes": sorted(gettel_txns["property_class"].dropna().unique().tolist()),
        "property_types":   sorted(gettel_txns["property_type"].dropna().unique().tolist()),
        "ownership_types":  sorted(gettel_txns["ownership_type"].dropna().unique().tolist()),
        "cities":           sorted(gettel_txns["city"].dropna().unique().tolist()),
        "years":            sorted([int(y) for y in gettel_txns["sale_year"].dropna().unique() if y]),
        "subdivisions":     sorted(gettel_txns["subdivision"].dropna().unique().tolist()),
    },
    "kpis": {
        "total_transactions":   len(txn_records),
        "total_volume":         sum(prices),
        "avg_deal_size":        sum(prices)/len(prices) if prices else 0,
        "median_deal_size":     float(pd.Series(prices).median()) if prices else 0,
        "unique_companies":     len(set(gettel_txns["vendor_entity"].tolist() + gettel_txns["purchaser_entity"].tolist())),
        "unique_persons":       len(set(gettel_txns["vendor_person"].tolist() + gettel_txns["purchaser_person"].tolist()) - {""}),
        "cores_matched_companies": len([v for v in vendor_cans.values() if v] + [v for v in purchaser_cans.values() if v]),
    }
}

with open("dashboard_data.json","w",encoding="utf-8") as f:
    json.dump(dashboard_data, f, default=str)
print(f"  dashboard_data.json: {len(txn_records)} transactions")

# ── 1F: SCRAPE LIST ───────────────────────────────────────────────────────────

print("Building gettel_scrape_list.csv...")

entity_txn_count = defaultdict(int)
for _, row in gettel_txns.iterrows():
    for side in ["vendor","purchaser"]:
        ent   = row[f"{side}_entity"]
        etype = row[f"{side}_entity_type"]
        can   = row[f"{side}_can"]
        if ent and etype == "company" and not can:
            entity_txn_count[ent] += 1

scrape_rows = []
for ent, cnt in entity_txn_count.items():
    norm = normalize(ent)
    prov = detect_province(ent)
    if prov in ("BC","ON","OTHER"):
        continue
    priority = "high" if cnt >= 5 else ("medium" if cnt >= 2 else "low")
    scrape_rows.append({
        "entity_name":     ent,
        "normalized_name": norm,
        "txn_count":       cnt,
        "entity_type":     "company",
        "province_guess":  prov,
        "priority":        priority,
    })

scrape_df = pd.DataFrame(scrape_rows).sort_values("txn_count", ascending=False)
scrape_df.to_csv("gettel_scrape_list.csv", index=False)
print(f"  Scrape list: {len(scrape_df)} companies")
print(f"    High priority (5+ txns): {(scrape_df.priority=='high').sum()}")
print(f"    Medium priority (2-4 txns): {(scrape_df.priority=='medium').sum()}")
print(f"    Low priority (1 txn): {(scrape_df.priority=='low').sum()}")

# ── 1G: SAVE PARSED TRANSACTIONS ─────────────────────────────────────────────

gettel_txns.to_csv("gettel_transactions_parsed.csv", index=False)
print("\nAll outputs saved:")
print("  graph_data.json")
print("  dashboard_data.json")
print("  gettel_scrape_list.csv")
print("  gettel_transactions_parsed.csv")


In [ ]:
# @title 2 — NETWORK MIND MAP (HTML)
# Input: graph_data.json
# Output: mindmap_section.html

import json

with open("graph_data.json","r",encoding="utf-8") as f:
    graph_data = f.read()

html = r"""
<style>
  @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600&family=DM+Mono:wght@400;500&display=swap');
  :root{--bg:#f8f8f6;--surface:#fff;--border:#e0e0da;--text:#1a1a1a;--muted:#6b6b6b;
    --accent:#111;--cores-col:#111;--gettel-col:#3a7bd5;--both-col:#6b21a8;
    --depth1:#111;--depth2:#444;--depth3:#777;--person:#fff;--other:#faf5eb;}
  *{box-sizing:border-box;margin:0;padding:0;}
  body{font-family:'DM Sans',sans-serif;background:var(--bg);color:var(--text);height:100vh;overflow:hidden;}
  .mm-app{display:grid;grid-template-columns:280px 1fr 320px;height:100vh;}

  .mm-sb{background:var(--surface);border-right:1px solid var(--border);display:flex;flex-direction:column;overflow:hidden;}
  .mm-sb-header{padding:14px 16px 0;border-bottom:1px solid var(--border);}
  .mm-sb-header h2{font-size:13px;font-weight:600;letter-spacing:.04em;text-transform:uppercase;color:var(--muted);margin-bottom:10px;}
  .mm-sb-tabs{display:flex;gap:0;border-bottom:1px solid var(--border);}
  .mm-sb-tab{flex:1;padding:8px 0;font-size:12px;font-weight:500;border:none;background:none;cursor:pointer;color:var(--muted);border-bottom:2px solid transparent;}
  .mm-sb-tab.active{color:var(--text);border-bottom-color:var(--accent);}
  .mm-filters{padding:10px 12px;border-bottom:1px solid var(--border);display:flex;flex-direction:column;gap:6px;}
  .mm-filters input,.mm-filters select{width:100%;padding:5px 8px;font-size:12px;font-family:inherit;border:1px solid var(--border);border-radius:4px;background:var(--bg);color:var(--text);}
  .mm-filters-row{display:flex;gap:6px;}
  .mm-filters-row input{flex:1;}
  .btn-reset{padding:4px 8px;font-size:11px;font-family:inherit;border:1px solid var(--border);border-radius:4px;background:none;cursor:pointer;color:var(--muted);}
  .btn-reset:hover{background:var(--bg);}
  .mm-list{flex:1;overflow-y:auto;padding:6px 0;}
  .mm-list-item{padding:8px 14px;cursor:pointer;border-left:3px solid transparent;font-size:12px;line-height:1.4;}
  .mm-list-item:hover{background:#f4f4f2;}
  .mm-list-item.active{border-left-color:var(--accent);background:#f4f4f2;}
  .mm-list-item .li-name{font-weight:500;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;}
  .mm-list-item .li-meta{color:var(--muted);font-size:11px;display:flex;gap:6px;margin-top:2px;}
  .mm-list-count{padding:6px 14px;font-size:11px;color:var(--muted);}

  .mm-canvas{position:relative;overflow:hidden;background:var(--bg);}
  #mm-svg{width:100%;height:100%;}
  .mm-ctrl{position:absolute;bottom:16px;right:16px;display:flex;flex-direction:column;gap:4px;}
  .mm-ctrl button{width:32px;height:32px;border:1px solid var(--border);background:var(--surface);border-radius:4px;cursor:pointer;font-size:14px;display:flex;align-items:center;justify-content:center;}
  .mm-ctrl button:hover{background:var(--bg);}
  .mm-empty{position:absolute;top:50%;left:50%;transform:translate(-50%,-50%);text-align:center;color:var(--muted);}
  .mm-empty .e-icon{font-size:40px;margin-bottom:8px;}
  .mm-empty p{font-size:13px;}
  .mm-legend{position:absolute;bottom:16px;left:16px;background:var(--surface);border:1px solid var(--border);border-radius:6px;padding:8px 12px;font-size:11px;}
  .mm-legend-row{display:flex;align-items:center;gap:6px;margin-bottom:4px;}
  .mm-legend-row:last-child{margin-bottom:0;}
  .l-dot{width:10px;height:10px;border-radius:50%;display:inline-block;flex-shrink:0;}

  .mm-detail{background:var(--surface);border-left:1px solid var(--border);overflow-y:auto;font-size:12px;}
  .mm-detail-empty{display:flex;align-items:center;justify-content:center;height:100%;color:var(--muted);font-size:13px;}
  .mm-detail-inner{padding:16px;}
  .d-name{font-size:15px;font-weight:600;line-height:1.3;margin-bottom:8px;}
  .d-badges{display:flex;flex-wrap:wrap;gap:4px;margin-bottom:12px;}
  .badge{padding:2px 7px;border-radius:10px;font-size:10px;font-weight:600;letter-spacing:.03em;text-transform:uppercase;}
  .badge-active{background:#d1fae5;color:#065f46;}
  .badge-dissolved{background:#fee2e2;color:#991b1b;}
  .badge-cores{background:#111;color:#fff;}
  .badge-gettel{background:#dbeafe;color:#1d4ed8;}
  .badge-both{background:#ede9fe;color:#5b21b6;}
  .badge-d1{background:#111;color:#fff;}
  .badge-d2{background:#444;color:#fff;}
  .badge-d3{background:#777;color:#fff;}
  .d-section{margin-top:14px;}
  .d-section-title{font-size:10px;font-weight:600;text-transform:uppercase;letter-spacing:.06em;color:var(--muted);border-bottom:1px solid var(--border);padding-bottom:4px;margin-bottom:8px;}
  .d-row{display:flex;gap:8px;margin-bottom:4px;}
  .d-row .lbl{color:var(--muted);min-width:90px;flex-shrink:0;}
  .d-row .val{font-weight:500;word-break:break-word;}
  .member-row{padding:6px 0;border-bottom:1px solid var(--border);cursor:pointer;}
  .member-row:last-child{border-bottom:none;}
  .member-row:hover{background:#fafaf8;}
  .member-name{font-weight:500;}
  .member-meta{color:var(--muted);font-size:11px;margin-top:1px;}
  .txn-row{padding:6px 0;border-bottom:1px solid var(--border);cursor:pointer;}
  .txn-row:last-child{border-bottom:none;}
  .txn-row:hover .txn-header{text-decoration:underline;}
  .txn-header{display:flex;justify-content:space-between;align-items:start;}
  .txn-price{font-weight:600;font-family:'DM Mono',monospace;white-space:nowrap;}
  .txn-desc{color:var(--muted);font-size:11px;margin-top:2px;}
  .txn-detail{display:none;background:#fafaf8;padding:8px;border-radius:4px;margin-top:4px;font-size:11px;}
  .txn-detail.open{display:block;}
  .txn-badge{font-size:10px;padding:1px 5px;border-radius:3px;font-weight:600;}
  .txn-vendor{background:#f0fdf4;color:#166534;}
  .txn-purchaser{background:#eff6ff;color:#1e40af;}

  .node circle{stroke:#333;stroke-width:1.5px;cursor:pointer;}
  .node text{font-family:'DM Sans',sans-serif;font-size:11px;pointer-events:none;}
  .node .txn-badge-svg{font-family:'DM Mono',monospace;font-size:9px;fill:#555;}
  .link{fill:none;stroke:#ccc;stroke-width:1px;}
  .collapsed-indicator{fill:#999;font-size:10px;cursor:pointer;}
</style>

<div id="mindmap-tab">
<div class="mm-app">

  <div class="mm-sb">
    <div class="mm-sb-header">
      <h2>CORES + GETTEL Network</h2>
      <div class="mm-sb-tabs">
        <button class="mm-sb-tab active" data-tab="companies" onclick="MM_switchTab('companies')">Companies</button>
        <button class="mm-sb-tab" data-tab="people" onclick="MM_switchTab('people')">People</button>
      </div>
    </div>

    <div id="mm-co-filters" class="mm-filters">
      <input id="mm-f-co" placeholder="Search company…" oninput="MM_filterCompanies()">
      <div class="mm-filters-row">
        <select id="mm-f-status" onchange="MM_filterCompanies()">
          <option value="">All statuses</option>
          <option>Active</option><option>Dissolved</option>
        </select>
        <select id="mm-f-source" onchange="MM_filterCompanies()">
          <option value="">All sources</option>
          <option value="cores">CORES</option>
          <option value="gettel_only">Gettel Only</option>
          <option value="cores+gettel">Both</option>
        </select>
      </div>
      <div class="mm-filters-row">
        <select id="mm-f-depth" onchange="MM_filterCompanies()">
          <option value="">All depths</option>
          <option value="1">Depth 1</option>
          <option value="2">Depth 2</option>
          <option value="3">Depth 3</option>
        </select>
        <input id="mm-f-mintxn" type="number" min="0" placeholder="Min txns" oninput="MM_filterCompanies()">
      </div>
      <button class="btn-reset" onclick="MM_resetCoFilters()">Reset</button>
    </div>

    <div id="mm-pe-filters" class="mm-filters" style="display:none">
      <input id="mm-f-person" placeholder="Search person…" oninput="MM_filterPeople()">
      <div class="mm-filters-row">
        <select id="mm-f-role" onchange="MM_filterPeople()">
          <option value="">All roles</option>
          <option>Director</option><option>Shareholder</option>
        </select>
        <input id="mm-f-mincos" type="number" min="0" placeholder="Min cos" oninput="MM_filterPeople()">
      </div>
      <button class="btn-reset" onclick="MM_resetPeFilters()">Reset</button>
    </div>

    <div id="mm-list-count" class="mm-list-count"></div>
    <div id="mm-co-list" class="mm-list"></div>
    <div id="mm-pe-list" class="mm-list" style="display:none"></div>
  </div>

  <div class="mm-canvas" id="mm-canvas">
    <svg id="mm-svg"></svg>
    <div class="mm-empty" id="mm-empty">
      <div class="e-icon">🏢</div>
      <p>Select a company to explore its network</p>
    </div>
    <div class="mm-ctrl">
      <button title="Zoom in" onclick="MM_zoom(1.3)">+</button>
      <button title="Zoom out" onclick="MM_zoom(0.77)">−</button>
      <button title="Fit" onclick="MM_fit()" style="font-size:11px">Fit</button>
      <button title="Expand all" onclick="MM_expandAll()" style="font-size:11px">Exp</button>
      <button title="Collapse all" onclick="MM_collapseAll()" style="font-size:11px">Col</button>
    </div>
    <div class="mm-legend">
      <div class="mm-legend-row"><span class="l-dot" style="background:#111"></span>CORES company</div>
      <div class="mm-legend-row"><span class="l-dot" style="background:#3a7bd5"></span>Gettel only</div>
      <div class="mm-legend-row"><span class="l-dot" style="background:#fff;border:1.5px solid #111"></span>Individual</div>
      <div class="mm-legend-row"><span class="l-dot" style="background:#faf5eb;border:1.5px solid #aaa"></span>Other (trust/jointly)</div>
    </div>
  </div>

  <div class="mm-detail" id="mm-detail">
    <div class="mm-detail-empty">Click a node to see details</div>
  </div>

</div>
</div>

<script>
(function(){
const MM_DATA = """ + graph_data + r""";

const CI = MM_DATA.company_info;
const CL = MM_DATA.company_list;
const PI = MM_DATA.person_index;

let MM_activeTab = 'companies';
let MM_selectedCAN = null;
let MM_collapsed = new Set();
let MM_svg, MM_g, MM_zoomBeh;
let MM_currentD3Data = null;
let MM_filteredCos = [...CL];
let MM_filteredPe = Object.entries(PI);

function fmtPrice(v){
  if(!v) return '—';
  if(v>=1e9) return '$'+(v/1e9).toFixed(2)+'B';
  if(v>=1e6) return '$'+(v/1e6).toFixed(1)+'M';
  if(v>=1e3) return '$'+(v/1e3).toFixed(0)+'K';
  return '$'+v.toLocaleString();
}

window.MM_switchTab = function(tab){
  MM_activeTab = tab;
  document.querySelectorAll('.mm-sb-tab').forEach(b=>b.classList.toggle('active', b.dataset.tab===tab));
  document.getElementById('mm-co-filters').style.display = tab==='companies'?'':'none';
  document.getElementById('mm-pe-filters').style.display = tab==='people'?'':'none';
  document.getElementById('mm-co-list').style.display    = tab==='companies'?'':'none';
  document.getElementById('mm-pe-list').style.display    = tab==='people'?'':'none';
  if(tab==='companies') renderCoList();
  else renderPeList();
};

window.MM_filterCompanies = function(){
  const q    = document.getElementById('mm-f-co').value.toLowerCase();
  const stat = document.getElementById('mm-f-status').value;
  const src  = document.getElementById('mm-f-source').value;
  const dep  = document.getElementById('mm-f-depth').value;
  const minT = parseInt(document.getElementById('mm-f-mintxn').value)||0;
  MM_filteredCos = CL.filter(c=>{
    if(q && !c.label.toLowerCase().includes(q)) return false;
    if(stat && c.status !== stat) return false;
    if(src && c.source !== src) return false;
    if(dep && String(c.depth) !== dep) return false;
    if(c.txn_count < minT) return false;
    return true;
  });
  renderCoList();
};

window.MM_resetCoFilters = function(){
  ['mm-f-co','mm-f-mintxn'].forEach(id=>document.getElementById(id).value='');
  ['mm-f-status','mm-f-source','mm-f-depth'].forEach(id=>document.getElementById(id).value='');
  MM_filterCompanies();
};

window.MM_filterPeople = function(){
  const q    = document.getElementById('mm-f-person').value.toLowerCase();
  const role = document.getElementById('mm-f-role').value.toLowerCase();
  const minC = parseInt(document.getElementById('mm-f-mincos').value)||0;
  MM_filteredPe = Object.entries(PI).filter(([k,p])=>{
    if(q && !p.label.toLowerCase().includes(q)) return false;
    if(minC && p.companies.length < minC) return false;
    return true;
  });
  renderPeList();
};

window.MM_resetPeFilters = function(){
  ['mm-f-person','mm-f-mincos'].forEach(id=>document.getElementById(id).value='');
  document.getElementById('mm-f-role').value='';
  MM_filterPeople();
};

function renderCoList(){
  const el = document.getElementById('mm-co-list');
  document.getElementById('mm-list-count').textContent = MM_filteredCos.length+' companies';
  const CHUNK = 100;
  let rendered = 0;
  el.innerHTML='';
  function addChunk(){
    const slice = MM_filteredCos.slice(rendered, rendered+CHUNK);
    slice.forEach(c=>{
      const div = document.createElement('div');
      div.className='mm-list-item'+(c.can===MM_selectedCAN?' active':'');
      div.innerHTML=`<div class="li-name">${c.label||c.can}</div>
        <div class="li-meta">
          <span>${c.status||'—'}</span>
          <span>${sourceBadgeText(c.source)}</span>
          ${c.txn_count>0?`<span>📊 ${c.txn_count}</span>`:''}
        </div>`;
      div.onclick=()=>MM_selectCompany(c.can);
      el.appendChild(div);
    });
    rendered+=slice.length;
    if(rendered<MM_filteredCos.length){
      el.addEventListener('scroll',function onScroll(){
        if(el.scrollTop+el.clientHeight > el.scrollHeight-40){
          el.removeEventListener('scroll',onScroll);
          addChunk();
        }
      },{once:true});
    }
  }
  addChunk();
}

function renderPeList(){
  const el = document.getElementById('mm-pe-list');
  document.getElementById('mm-list-count').textContent = MM_filteredPe.length+' people';
  el.innerHTML='';
  MM_filteredPe.slice(0,200).forEach(([key,p])=>{
    const div = document.createElement('div');
    div.className='mm-list-item';
    div.innerHTML=`<div class="li-name">${p.label}</div>
      <div class="li-meta">
        <span>${p.companies.length} co${p.companies.length!==1?'s':''}</span>
        <span>${p.transactions.length} txns</span>
      </div>`;
    div.onclick=()=>MM_selectPerson(key);
    el.appendChild(div);
  });
}

function sourceBadgeText(s){
  if(s==='cores') return 'CORES';
  if(s==='gettel_only') return 'GETTEL';
  if(s==='cores+gettel') return 'CORES+GETTEL';
  return s||'';
}

// ─── D3 TREE ────────────────────────────────────────────────────────────────
function initSvg(){
  const canvas = document.getElementById('mm-canvas');
  MM_svg = d3.select('#mm-svg');
  MM_svg.selectAll('*').remove();
  MM_zoomBeh = d3.zoom().scaleExtent([0.05,4]).on('zoom',e=>MM_g.attr('transform',e.transform));
  MM_svg.call(MM_zoomBeh);
  MM_g = MM_svg.append('g');
}

function buildHierarchy(can, visited){
  visited = visited || new Set();
  if(visited.has(can)) return null;
  visited.add(can);
  const co = CI[can];
  if(!co) return null;
  const node = {
    id: can, label: co.label||can, nodeType:'company',
    source: co.source, depth: co.depth, status: co.status,
    txnCount: (co.transactions||[]).length,
    children: [],
  };
  if(!MM_collapsed.has(can)){
    (co.children||[]).forEach(ch=>{
      if(ch.node_type==='legal entity'||ch.node_type==='legal_entity'){
        const childCAN = ch.corp_can;
        if(childCAN && CI[childCAN]){
          const sub = buildHierarchy(childCAN, visited);
          if(sub) node.children.push(sub);
        } else {
          node.children.push({id:ch.id, label:ch.label, nodeType:'company-ext',
            source:'gettel_only', depth:'-', status:'', txnCount:0, children:[]});
        }
      } else {
        node.children.push({id:ch.id, label:ch.label,
          nodeType: ch.node_type==='individual'?'person':'other',
          role:ch.role, pct:ch.pct_shares, status:ch.status,
          appointment:ch.appointment, address:ch.address, children:[]});
      }
    });
  }
  return node;
}

function drawTree(can){
  if(!MM_svg) initSvg();
  document.getElementById('mm-empty').style.display='none';
  const hierarchy = buildHierarchy(can);
  if(!hierarchy) return;
  MM_currentD3Data = {can};

  const root = d3.hierarchy(hierarchy);
  const treeLayout = d3.tree().nodeSize([34,240]);
  treeLayout(root);

  MM_g.selectAll('*').remove();

  MM_g.selectAll('.link')
    .data(root.links())
    .enter().append('path')
    .attr('class','link')
    .attr('d', d3.linkHorizontal().x(d=>d.y).y(d=>d.x));

  const node = MM_g.selectAll('.node')
    .data(root.descendants())
    .enter().append('g')
    .attr('class','node')
    .attr('transform',d=>`translate(${d.y},${d.x})`)
    .on('click',(_,d)=>MM_nodeClick(d.data));

  node.append('circle')
    .attr('r', d=>d.data.nodeType==='company'||d.data.nodeType==='company-ext'?9:6)
    .style('fill', d=>{
      const t = d.data.nodeType;
      if(t==='person') return '#fff';
      if(t==='other') return '#faf5eb';
      const s = d.data.source;
      if(s==='gettel_only') return '#3a7bd5';
      if(s==='cores+gettel') return '#6b21a8';
      const dep = String(d.data.depth);
      if(dep==='1') return '#111';
      if(dep==='2') return '#444';
      return '#777';
    })
    .style('stroke', d=>{
      if(d.data.nodeType==='person') return '#111';
      if(d.data.nodeType==='other') return '#aaa';
      return '#333';
    });

  node.append('text')
    .attr('dy','0.31em')
    .attr('x', d=>{
      const isCompany = d.data.nodeType==='company'||d.data.nodeType==='company-ext';
      return (d.children&&d.children.length>0) ? (isCompany?-13:-10) : (isCompany?13:10);
    })
    .attr('text-anchor', d=>(d.children&&d.children.length>0)?'end':'start')
    .text(d=>d.data.label.length>28?d.data.label.slice(0,26)+'…':d.data.label);

  node.filter(d=>(d.data.nodeType==='company'||d.data.nodeType==='company-ext') && d.data.txnCount>0)
    .append('text')
    .attr('class','txn-badge-svg')
    .attr('dy','-1em')
    .attr('x',0)
    .attr('text-anchor','middle')
    .text(d=>`[${d.data.txnCount}]`);

  // Collapsed indicator
  node.filter(d=>MM_collapsed.has(d.data.id))
    .append('text')
    .attr('class','collapsed-indicator')
    .attr('dy','0.31em').attr('x',13).attr('text-anchor','start')
    .text('▶');

  MM_fit();
}

window.MM_selectCompany = function(can){
  MM_selectedCAN = can;
  renderCoList();
  drawTree(can);
  showCompanyDetail(can);
};

window.MM_selectPerson = function(key){
  showPersonDetail(key);
};

function MM_nodeClick(data){
  if(data.nodeType==='company'||data.nodeType==='company-ext'){
    if(MM_collapsed.has(data.id)) MM_collapsed.delete(data.id);
    else MM_collapsed.add(data.id);
    drawTree(MM_selectedCAN);
    showCompanyDetail(data.id);
  } else {
    showPersonDetailByLabel(data.label);
  }
}

window.MM_zoom = function(factor){
  MM_svg.transition().call(MM_zoomBeh.scaleBy, factor);
};

window.MM_fit = function(){
  const canvas = document.getElementById('mm-canvas');
  const w = canvas.clientWidth, h = canvas.clientHeight;
  const bbox = MM_g.node().getBBox();
  if(!bbox||bbox.width===0) return;
  const scale = Math.min(0.9, Math.min(w/bbox.width, h/bbox.height)*0.85);
  const tx = (w-bbox.width*scale)/2 - bbox.x*scale;
  const ty = (h-bbox.height*scale)/2 - bbox.y*scale;
  MM_svg.transition().duration(400).call(MM_zoomBeh.transform, d3.zoomIdentity.translate(tx,ty).scale(scale));
};

window.MM_expandAll = function(){
  MM_collapsed.clear();
  if(MM_selectedCAN) drawTree(MM_selectedCAN);
};

window.MM_collapseAll = function(){
  if(!MM_selectedCAN) return;
  const co = CI[MM_selectedCAN];
  if(co)(co.children||[]).forEach(ch=>{if(ch.corp_can)MM_collapsed.add(ch.corp_can);});
  drawTree(MM_selectedCAN);
};

// ─── DETAIL PANEL ────────────────────────────────────────────────────────────
function showCompanyDetail(can){
  const co = CI[can];
  const el = document.getElementById('mm-detail');
  if(!co){el.innerHTML='<div class="mm-detail-empty">No data for this company.</div>';return;}

  const srcBadge = co.source==='cores'?'<span class="badge badge-cores">CORES</span>'
    : co.source==='gettel_only'?'<span class="badge badge-gettel">GETTEL</span>'
    : '<span class="badge badge-both">CORES+GETTEL</span>';
  const statBadge = co.status==='Active'?'<span class="badge badge-active">Active</span>'
    : co.status?`<span class="badge badge-dissolved">${co.status}</span>`:'';
  const depBadge = co.depth?`<span class="badge badge-d${co.depth}">Depth ${co.depth}</span>`:'';

  let html = `<div class="mm-detail-inner">
    <div class="d-name">${co.label||can}</div>
    <div class="d-badges">${statBadge}${srcBadge}${depBadge}</div>`;

  if(co.source!=='gettel_only'){
    html+=`<div class="d-section"><div class="d-section-title">Registration</div>
      <div class="d-row"><span class="lbl">CAN</span><span class="val">${co.can}</span></div>
      <div class="d-row"><span class="lbl">Type</span><span class="val">${co.le_type||'—'}</span></div>
      <div class="d-row"><span class="lbl">Corp Type</span><span class="val">${co.corp_type||'—'}</span></div>
      <div class="d-row"><span class="lbl">Reg Date</span><span class="val">${co.registration_date||'—'}</span></div>
      <div class="d-row"><span class="lbl">Last AR</span><span class="val">${co.last_ar_year||'—'} (${co.last_ar_filed||'—'})</span></div>
    </div>`;
  }

  if(co.address||co.email){
    html+=`<div class="d-section"><div class="d-section-title">Address</div>
      <div class="d-row"><span class="lbl">Address</span><span class="val">${co.address||'—'}</span></div>
      ${co.email?`<div class="d-row"><span class="lbl">Email</span><span class="val">${co.email}</span></div>`:''}
      ${co.agent?`<div class="d-row"><span class="lbl">Agent</span><span class="val">${co.agent}</span></div>`:''}
    </div>`;
  }

  const members = co.children||[];
  if(members.length){
    html+=`<div class="d-section"><div class="d-section-title">Members (${members.length})</div>`;
    members.forEach(m=>{
      html+=`<div class="member-row">
        <div class="member-name">${m.label}</div>
        <div class="member-meta">${m.role}${m.pct_shares?' · '+m.pct_shares+'%':''}${m.status?' · '+m.status:''}</div>
      </div>`;
    });
    html+='</div>';
  }

  const txns = co.transactions||[];
  if(txns.length){
    html+=`<div class="d-section"><div class="d-section-title">Transactions (${txns.length})</div>`;
    txns.sort((a,b)=>b.sale_date>a.sale_date?1:-1).forEach((t,i)=>{
      const roleCls = t.role==='vendor'?'txn-vendor':'txn-purchaser';
      html+=`<div class="txn-row" onclick="MM_toggleTxn('txn-${i}-${can}')">
        <div class="txn-header">
          <span><span class="txn-badge ${roleCls}">${t.role.toUpperCase()}</span> ${t.sale_date||'—'}</span>
          <span class="txn-price">${fmtPrice(t.sale_price)}</span>
        </div>
        <div class="txn-desc">${t.description||''} · ${t.city||''}</div>
        <div class="txn-detail" id="txn-${i}-${can}">
          <div class="d-row"><span class="lbl">Address</span><span class="val">${t.address||'—'}</span></div>
          <div class="d-row"><span class="lbl">Class</span><span class="val">${t.property_class||'—'}</span></div>
          <div class="d-row"><span class="lbl">$/Unit</span><span class="val">${t.unit_price?fmtPrice(t.unit_price):'—'}</span></div>
          <div class="d-row"><span class="lbl">Bldg Area</span><span class="val">${t.bldg_area?t.bldg_area.toLocaleString()+' '+t.bldg_units:'—'}</span></div>
          <div class="d-row"><span class="lbl">Site Area</span><span class="val">${t.site_area?t.site_area.toLocaleString()+' '+t.site_units:'—'}</span></div>
          <div class="d-row"><span class="lbl">Year Built</span><span class="val">${t.year_built||'—'}</span></div>
          <div class="d-row"><span class="lbl">Counterparty</span><span class="val">${t.counterparty||'—'}${t.counterparty_person?' · '+t.counterparty_person:''}</span></div>
        </div>
      </div>`;
    });
    html+='</div>';
  }

  html+='</div>';
  el.innerHTML=html;
}

window.MM_toggleTxn = function(id){
  const el = document.getElementById(id);
  if(el) el.classList.toggle('open');
};

function showPersonDetail(key){
  const p = PI[key];
  const el = document.getElementById('mm-detail');
  if(!p){el.innerHTML='<div class="mm-detail-empty">Person not found.</div>';return;}
  let html = `<div class="mm-detail-inner">
    <div class="d-name">${p.label}</div>
    <div class="d-section"><div class="d-section-title">Companies (${p.companies.length})</div>`;
  p.companies.forEach(can=>{
    const co = CI[can];
    html+=`<div class="member-row" onclick="MM_selectCompany('${can}')">
      <div class="member-name">${co?co.label:can}</div>
      <div class="member-meta">${co?co.status||'':''}</div>
    </div>`;
  });
  html+='</div>';
  if(p.transactions.length){
    const byConf = {high_direct:[], medium_cores:[], low_name:[]};
    p.transactions.forEach(t=>(byConf[t.confidence]||[]).push(t));
    html+=`<div class="d-section"><div class="d-section-title">Transactions (${p.transactions.length})</div>`;
    ['high_direct','medium_cores','low_name'].forEach(conf=>{
      if(!byConf[conf].length) return;
      html+=`<div style="font-size:11px;color:var(--muted);margin:6px 0 3px;font-weight:600;">${conf.replace('_',' ').toUpperCase()}</div>`;
      byConf[conf].forEach(t=>{
        const txn = MM_DATA.company_info;
        const co = CI[t.company_can];
        html+=`<div class="txn-row">
          <div class="txn-header">
            <span>${co?co.label:t.company_can}</span>
            <span class="txn-badge ${t.role==='vendor'?'txn-vendor':'txn-purchaser'}">${(t.role||'').toUpperCase()}</span>
          </div>
        </div>`;
      });
    });
    html+='</div>';
  }
  html+='</div>';
  el.innerHTML=html;
}

function showPersonDetailByLabel(label){
  const key = Object.keys(PI).find(k=>PI[k].label===label);
  if(key) showPersonDetail(key);
  else{
    document.getElementById('mm-detail').innerHTML=
      `<div class="mm-detail-inner"><div class="d-name">${label}</div>
       <div style="color:var(--muted);font-size:12px;margin-top:8px;">Not in CORES person index.</div></div>`;
  }
}

// ─── INIT ────────────────────────────────────────────────────────────────────
MM_filteredCos = [...CL];
MM_filteredPe  = Object.entries(PI);
renderCoList();
initSvg();

})();
</script>
"""

with open("mindmap_section.html","w",encoding="utf-8") as f:
    f.write(html)

print("mindmap_section.html written.")
print(f"  File size: {len(html)/1024:.0f} KB (excluding embedded data)")


In [ ]:
# @title 3 — TRANSACTION DASHBOARD (HTML)
# Input: dashboard_data.json
# Output: dashboard_section.html

import json

with open("dashboard_data.json","r",encoding="utf-8") as f:
    dashboard_data = f.read()

html = r"""
<style>
  @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600&family=DM+Mono:wght@400;500&display=swap');
  #dashboard-tab *{box-sizing:border-box;margin:0;padding:0;}
  #dashboard-tab{font-family:'DM Sans',sans-serif;background:#f8f8f6;color:#1a1a1a;min-height:100vh;}

  .db-header{background:#fff;border-bottom:1px solid #e0e0da;padding:12px 20px;}
  .db-header h2{font-size:13px;font-weight:600;text-transform:uppercase;letter-spacing:.04em;color:#6b6b6b;margin-bottom:10px;}
  .db-kpis{display:flex;gap:16px;flex-wrap:wrap;}
  .db-kpi{background:#f8f8f6;border:1px solid #e0e0da;border-radius:6px;padding:8px 14px;min-width:120px;}
  .db-kpi .kv{font-size:20px;font-weight:600;font-family:'DM Mono',monospace;}
  .db-kpi .kl{font-size:11px;color:#6b6b6b;margin-top:2px;}

  .db-body{display:grid;grid-template-columns:220px 1fr;height:calc(100vh - 90px);}

  .db-sidebar{background:#fff;border-right:1px solid #e0e0da;overflow-y:auto;padding:14px 12px;}
  .db-sidebar h3{font-size:11px;font-weight:600;text-transform:uppercase;letter-spacing:.05em;color:#6b6b6b;margin:12px 0 6px;}
  .db-sidebar h3:first-child{margin-top:0;}
  .db-cb-group{display:flex;flex-direction:column;gap:4px;}
  .db-cb-label{display:flex;align-items:center;gap:6px;font-size:12px;cursor:pointer;}
  .db-cb-label input{cursor:pointer;}
  .db-range-row{display:flex;gap:6px;align-items:center;font-size:11px;margin-bottom:4px;}
  .db-range-input{width:80px;padding:4px 6px;font-size:11px;font-family:inherit;border:1px solid #e0e0da;border-radius:4px;background:#f8f8f6;}
  .db-text-input{width:100%;padding:5px 8px;font-size:12px;font-family:inherit;border:1px solid #e0e0da;border-radius:4px;background:#f8f8f6;margin-bottom:4px;}
  .db-select{width:100%;padding:5px 8px;font-size:12px;font-family:inherit;border:1px solid #e0e0da;border-radius:4px;background:#f8f8f6;margin-bottom:4px;}
  .db-reset-btn{width:100%;margin-top:10px;padding:7px;font-size:12px;font-family:inherit;border:1px solid #e0e0da;border-radius:4px;background:none;cursor:pointer;color:#6b6b6b;}
  .db-reset-btn:hover{background:#f4f4f2;}
  .db-cores-row{display:flex;align-items:center;gap:6px;font-size:12px;cursor:pointer;margin-bottom:4px;}

  .db-main{overflow-y:auto;padding:16px;}
  .db-charts-row{display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:12px;}
  .db-chart-card{background:#fff;border:1px solid #e0e0da;border-radius:6px;padding:12px;resize:vertical;overflow:auto;min-height:200px;}
  .db-chart-card h4{font-size:11px;font-weight:600;text-transform:uppercase;letter-spacing:.04em;color:#6b6b6b;margin-bottom:10px;}
  .db-chart-wrap{position:relative;height:220px;}

  .db-table-card{background:#fff;border:1px solid #e0e0da;border-radius:6px;overflow:hidden;}
  .db-table-header{padding:10px 14px;border-bottom:1px solid #e0e0da;display:flex;align-items:center;justify-content:space-between;}
  .db-table-header h4{font-size:11px;font-weight:600;text-transform:uppercase;letter-spacing:.04em;color:#6b6b6b;}
  .db-export-btn{padding:4px 10px;font-size:11px;font-family:inherit;border:1px solid #e0e0da;border-radius:4px;background:none;cursor:pointer;}
  .db-table-wrap{overflow-x:auto;}
  table{width:100%;border-collapse:collapse;font-size:12px;}
  th{padding:8px 10px;text-align:left;border-bottom:2px solid #e0e0da;font-weight:600;font-size:11px;text-transform:uppercase;letter-spacing:.03em;color:#6b6b6b;cursor:pointer;white-space:nowrap;user-select:none;}
  th:hover{color:#111;}
  th.sort-asc::after{content:' ↑';}
  th.sort-desc::after{content:' ↓';}
  td{padding:7px 10px;border-bottom:1px solid #f4f4f2;vertical-align:top;}
  tr:hover td{background:#fafaf8;}
  tr.expanded td{background:#f8f8f6;}
  .db-price{font-family:'DM Mono',monospace;white-space:nowrap;}
  .db-co-matched{color:#111;font-weight:600;}
  .db-pagination{padding:10px 14px;border-top:1px solid #e0e0da;display:flex;align-items:center;gap:8px;font-size:12px;color:#6b6b6b;}
  .db-pagination button{padding:4px 10px;font-size:11px;font-family:inherit;border:1px solid #e0e0da;border-radius:4px;background:none;cursor:pointer;}
  .db-pagination button:disabled{opacity:.4;cursor:default;}
  .db-row-detail{display:none;background:#f8f8f6;padding:10px;font-size:11px;}
  .db-row-detail.open{display:table-row;}
  .db-row-detail td{padding:10px;color:#6b6b6b;}
  .db-detail-grid{display:grid;grid-template-columns:1fr 1fr;gap:6px 16px;}
  .db-detail-row{display:flex;gap:6px;}
  .db-detail-row .dl{color:#999;min-width:100px;}
  .db-detail-row .dv{font-weight:500;}
</style>

<div id="dashboard-tab">

  <div class="db-header">
    <h2>CORES + GETTEL Transaction Dashboard</h2>
    <div class="db-kpis">
      <div class="db-kpi"><div class="kv" id="db-k-txns">—</div><div class="kl">Transactions</div></div>
      <div class="db-kpi"><div class="kv" id="db-k-vol">—</div><div class="kl">Total Volume</div></div>
      <div class="db-kpi"><div class="kv" id="db-k-avg">—</div><div class="kl">Avg Deal Size</div></div>
      <div class="db-kpi"><div class="kv" id="db-k-matched">—</div><div class="kl">CORES Matches</div></div>
    </div>
  </div>

  <div class="db-body">

    <div class="db-sidebar" id="db-sidebar">
      <h3>Property Class</h3>
      <div class="db-cb-group" id="db-f-class"></div>

      <h3>Property Type</h3>
      <div class="db-cb-group" id="db-f-type"></div>

      <h3>Ownership Type</h3>
      <div class="db-cb-group" id="db-f-own"></div>

      <h3>City</h3>
      <select class="db-select" id="db-f-city"><option value="">All Cities</option></select>

      <h3>Year Range</h3>
      <div class="db-range-row">
        <input class="db-range-input" id="db-f-year-from" type="number" placeholder="From">
        <span>–</span>
        <input class="db-range-input" id="db-f-year-to"   type="number" placeholder="To">
      </div>

      <h3>Price Range ($)</h3>
      <div class="db-range-row">
        <input class="db-range-input" id="db-f-price-from" type="number" placeholder="Min">
        <span>–</span>
        <input class="db-range-input" id="db-f-price-to"   type="number" placeholder="Max">
      </div>

      <h3>Subdivision</h3>
      <select class="db-select" id="db-f-subdiv"><option value="">All Subdivisions</option></select>

      <h3>Company Search</h3>
      <input class="db-text-input" id="db-f-company" placeholder="Search vendor or purchaser…">

      <h3>Person Search</h3>
      <input class="db-text-input" id="db-f-person" placeholder="Search director/shareholder…">

      <label class="db-cores-row">
        <input type="checkbox" id="db-f-cores-only"> CORES matches only
      </label>

      <button class="db-reset-btn" onclick="DB_resetAll()">Reset All Filters</button>
    </div>

    <div class="db-main">
      <div class="db-charts-row">
        <div class="db-chart-card">
          <h4>Volume by Year</h4>
          <div class="db-chart-wrap"><canvas id="db-chart-year"></canvas></div>
        </div>
        <div class="db-chart-card">
          <h4>By Property Class</h4>
          <div class="db-chart-wrap"><canvas id="db-chart-class"></canvas></div>
        </div>
      </div>
      <div class="db-charts-row">
        <div class="db-chart-card">
          <h4>Top 20 Companies</h4>
          <div class="db-chart-wrap" style="height:260px"><canvas id="db-chart-cos"></canvas></div>
        </div>
        <div class="db-chart-card">
          <h4>Top 20 People</h4>
          <div class="db-chart-wrap" style="height:260px"><canvas id="db-chart-people"></canvas></div>
        </div>
      </div>

      <div class="db-table-card">
        <div class="db-table-header">
          <h4 id="db-table-title">Transactions</h4>
          <button class="db-export-btn" onclick="DB_exportCSV()">Export CSV</button>
        </div>
        <div class="db-table-wrap">
          <table id="db-table">
            <thead><tr>
              <th style="width:36px">#</th>
              <th data-col="sale_date" onclick="DB_sort('sale_date')">Date</th>
              <th data-col="city" onclick="DB_sort('city')">City</th>
              <th data-col="property_class" onclick="DB_sort('property_class')">Class</th>
              <th data-col="property_type" onclick="DB_sort('property_type')">Type</th>
              <th>Description</th>
              <th>Vendor</th>
              <th>Purchaser</th>
              <th data-col="sale_price" onclick="DB_sort('sale_price')">Price</th>
              <th data-col="unit_price" onclick="DB_sort('unit_price')">$/Unit</th>
              <th data-col="bldg_area" onclick="DB_sort('bldg_area')">Area</th>
            </tr></thead>
            <tbody id="db-tbody"></tbody>
          </table>
        </div>
        <div class="db-pagination">
          <button id="db-pg-prev" onclick="DB_page(-1)">← Prev</button>
          <span id="db-pg-info"></span>
          <button id="db-pg-next" onclick="DB_page(1)">Next →</button>
        </div>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
const DB_DATA = """ + dashboard_data + r""";

const TXNS = DB_DATA.transactions;
const FOPTS = DB_DATA.filter_options;
const KPI0  = DB_DATA.kpis;

let DB_filtered = [...TXNS];
let DB_page_num = 1;
const DB_PAGE_SIZE = 50;
let DB_sort_col = 'sale_date';
let DB_sort_dir = 'desc';

let DB_charts = {};

function fmtPrice(v){
  if(v==null||v===''||v===undefined) return '—';
  v = parseFloat(v);
  if(isNaN(v)) return '—';
  if(v>=1e9) return '$'+(v/1e9).toFixed(2)+'B';
  if(v>=1e6) return '$'+(v/1e6).toFixed(1)+'M';
  if(v>=1e3) return '$'+(v/1e3).toFixed(0)+'K';
  return '$'+v.toLocaleString();
}

function fmtArea(v,u){return v?(parseFloat(v)||0).toLocaleString()+' '+(u||''):'—';}

// ─── INIT FILTERS ────────────────────────────────────────────────────────────
function buildCBGroup(containerId, options, onChange){
  const c = document.getElementById(containerId);
  options.forEach(opt=>{
    const lbl = document.createElement('label');
    lbl.className='db-cb-label';
    lbl.innerHTML=`<input type="checkbox" value="${opt}" checked> ${opt}`;
    lbl.querySelector('input').addEventListener('change', onChange);
    c.appendChild(lbl);
  });
}

buildCBGroup('db-f-class', FOPTS.property_classes, DB_applyFilters);
buildCBGroup('db-f-type',  FOPTS.property_types,   DB_applyFilters);
buildCBGroup('db-f-own',   FOPTS.ownership_types,  DB_applyFilters);

const cityEl = document.getElementById('db-f-city');
FOPTS.cities.sort().forEach(c=>{const o=document.createElement('option');o.value=c;o.textContent=c;cityEl.appendChild(o);});
cityEl.addEventListener('change', DB_applyFilters);

const subdivEl = document.getElementById('db-f-subdiv');
FOPTS.subdivisions.sort().forEach(s=>{const o=document.createElement('option');o.value=s;o.textContent=s;subdivEl.appendChild(o);});
subdivEl.addEventListener('change', DB_applyFilters);

['db-f-year-from','db-f-year-to','db-f-price-from','db-f-price-to','db-f-company','db-f-person'].forEach(id=>{
  document.getElementById(id).addEventListener('input', DB_applyFilters);
});
document.getElementById('db-f-cores-only').addEventListener('change', DB_applyFilters);

// Set default year bounds
const years = FOPTS.years.filter(y=>y);
if(years.length){
  document.getElementById('db-f-year-from').placeholder = Math.min(...years);
  document.getElementById('db-f-year-to').placeholder   = Math.max(...years);
}

// ─── FILTER LOGIC ────────────────────────────────────────────────────────────
function getChecked(containerId){
  return [...document.querySelectorAll(`#${containerId} input[type=checkbox]:checked`)].map(el=>el.value);
}

function DB_applyFilters(){
  const classes  = getChecked('db-f-class');
  const types    = getChecked('db-f-type');
  const owns     = getChecked('db-f-own');
  const city     = document.getElementById('db-f-city').value;
  const subdiv   = document.getElementById('db-f-subdiv').value;
  const yFrom    = parseInt(document.getElementById('db-f-year-from').value)||0;
  const yTo      = parseInt(document.getElementById('db-f-year-to').value)||9999;
  const pFrom    = parseFloat(document.getElementById('db-f-price-from').value)||0;
  const pTo      = parseFloat(document.getElementById('db-f-price-to').value)||Infinity;
  const coQ      = document.getElementById('db-f-company').value.toLowerCase();
  const peQ      = document.getElementById('db-f-person').value.toLowerCase();
  const coresOnly= document.getElementById('db-f-cores-only').checked;

  DB_filtered = TXNS.filter(t=>{
    if(!classes.includes(t.property_class)) return false;
    if(!types.includes(t.property_type))   return false;
    if(owns.length && !owns.includes(t.ownership_type)) return false;
    if(city   && t.city !== city)           return false;
    if(subdiv && t.subdivision !== subdiv)  return false;
    const yr = parseInt(t.sale_year)||0;
    if(yFrom && yr < yFrom) return false;
    if(yTo<9999 && yr > yTo) return false;
    const pr = parseFloat(t.sale_price)||0;
    if(pFrom && pr < pFrom) return false;
    if(pTo<Infinity && pr > pTo) return false;
    if(coQ && !(t.vendor_entity||'').toLowerCase().includes(coQ) &&
             !(t.purchaser_entity||'').toLowerCase().includes(coQ)) return false;
    if(peQ && !(t.vendor_person||'').toLowerCase().includes(peQ) &&
             !(t.purchaser_person||'').toLowerCase().includes(peQ)) return false;
    if(coresOnly && !t.vendor_can && !t.purchaser_can) return false;
    return true;
  });

  DB_page_num = 1;
  updateKPIs();
  updateCharts();
  renderTable();
}
window.DB_applyFilters = DB_applyFilters;

function updateKPIs(){
  const prices  = DB_filtered.map(t=>parseFloat(t.sale_price)).filter(v=>!isNaN(v)&&v>0);
  const vol     = prices.reduce((s,v)=>s+v,0);
  const avg     = prices.length ? vol/prices.length : 0;
  const matched = DB_filtered.filter(t=>t.vendor_can||t.purchaser_can).length;
  document.getElementById('db-k-txns').textContent    = DB_filtered.length.toLocaleString();
  document.getElementById('db-k-vol').textContent     = fmtPrice(vol);
  document.getElementById('db-k-avg').textContent     = fmtPrice(avg);
  document.getElementById('db-k-matched').textContent = matched.toLocaleString();
  document.getElementById('db-table-title').textContent = `Transactions (${DB_filtered.length.toLocaleString()})`;
}

// ─── CHARTS ──────────────────────────────────────────────────────────────────
const GREY_PALETTE = ['#111','#333','#555','#777','#999','#bbb','#ccc','#ddd'];
const ACCENT_COLOR = '#3a7bd5';

function updateCharts(){
  updateYearChart();
  updateClassChart();
  updateCosChart();
  updatePeopleChart();
}

function updateYearChart(){
  const yearClass = {};
  DB_filtered.forEach(t=>{
    const y = t.sale_year||'Unknown';
    const c = t.property_class||'Unknown';
    if(!yearClass[y]) yearClass[y]={};
    yearClass[y][c] = (yearClass[y][c]||0)+(parseFloat(t.sale_price)||0);
  });
  const years   = Object.keys(yearClass).sort();
  const classes = [...new Set(DB_filtered.map(t=>t.property_class))].filter(Boolean);
  const datasets= classes.map((cls,i)=>({
    label:cls,
    data:years.map(y=>(yearClass[y]||{})[cls]||0),
    backgroundColor: GREY_PALETTE[i%GREY_PALETTE.length],
    stack:'s',
  }));

  if(DB_charts.year){
    DB_charts.year.data.labels=years;
    DB_charts.year.data.datasets=datasets;
    DB_charts.year.update();
  } else {
    DB_charts.year = new Chart(document.getElementById('db-chart-year'),{
      type:'bar',
      data:{labels:years,datasets},
      options:{responsive:true,maintainAspectRatio:false,
        plugins:{legend:{display:false},tooltip:{callbacks:{
          label:ctx=>`${ctx.dataset.label}: ${fmtPrice(ctx.raw)}`
        }}},
        scales:{x:{stacked:true},y:{stacked:true,ticks:{callback:v=>fmtPrice(v)}}},
      }
    });
  }
}

function updateClassChart(){
  const classTotals = {};
  DB_filtered.forEach(t=>{
    const c = t.property_class||'Unknown';
    classTotals[c]=(classTotals[c]||0)+(parseFloat(t.sale_price)||0);
  });
  const entries = Object.entries(classTotals).sort((a,b)=>b[1]-a[1]);
  const labels  = entries.map(e=>e[0]);
  const data    = entries.map(e=>e[1]);

  if(DB_charts.cls){
    DB_charts.cls.data.labels=labels;
    DB_charts.cls.data.datasets[0].data=data;
    DB_charts.cls.update();
  } else {
    DB_charts.cls = new Chart(document.getElementById('db-chart-class'),{
      type:'doughnut',
      data:{labels,datasets:[{data,backgroundColor:GREY_PALETTE}]},
      options:{responsive:true,maintainAspectRatio:false,
        plugins:{legend:{position:'right',labels:{font:{size:11}}},
          tooltip:{callbacks:{label:ctx=>`${ctx.label}: ${fmtPrice(ctx.raw)} (${(ctx.raw/data.reduce((s,v)=>s+v,0)*100).toFixed(1)}%)`}}}
      }
    });
  }
}

function updateCosChart(){
  const coCount = {};
  DB_filtered.forEach(t=>{
    const ve = t.vendor_entity; const pe = t.purchaser_entity;
    if(ve) coCount[ve]=(coCount[ve]||{count:0,matched:!!t.vendor_can}).count++||0, coCount[ve].count++;
    if(pe) coCount[pe]=(coCount[pe]||{count:0,matched:!!t.purchaser_can}).count++||0, coCount[pe].count++;
  });
  const sorted = Object.entries(coCount).sort((a,b)=>b[1].count-a[1].count).slice(0,20);
  const labels = sorted.map(e=>e[0].length>30?e[0].slice(0,28)+'…':e[0]);
  const data   = sorted.map(e=>e[1].count);
  const colors = sorted.map(e=>e[1].matched?'#111':'#aaa');

  if(DB_charts.cos){
    DB_charts.cos.data.labels=labels;
    DB_charts.cos.data.datasets[0].data=data;
    DB_charts.cos.data.datasets[0].backgroundColor=colors;
    DB_charts.cos.update();
  } else {
    DB_charts.cos = new Chart(document.getElementById('db-chart-cos'),{
      type:'bar',
      data:{labels,datasets:[{data,backgroundColor:colors,borderWidth:0}]},
      options:{indexAxis:'y',responsive:true,maintainAspectRatio:false,
        plugins:{legend:{display:false}},
        scales:{x:{ticks:{stepSize:1}},y:{ticks:{font:{size:10}}}},
      }
    });
  }
}

function updatePeopleChart(){
  const peCount = {};
  DB_filtered.forEach(t=>{
    const vp = t.vendor_person; const pp = t.purchaser_person;
    if(vp) peCount[vp]=(peCount[vp]||{count:0,cos:new Set()}), peCount[vp].count++, peCount[vp].cos.add(t.vendor_entity);
    if(pp) peCount[pp]=(peCount[pp]||{count:0,cos:new Set()}), peCount[pp].count++, peCount[pp].cos.add(t.purchaser_entity);
  });
  const sorted = Object.entries(peCount).sort((a,b)=>b[1].count-a[1].count).slice(0,20);
  const labels = sorted.map(e=>e[0]);
  const data   = sorted.map(e=>e[1].count);

  if(DB_charts.ppl){
    DB_charts.ppl.data.labels=labels;
    DB_charts.ppl.data.datasets[0].data=data;
    DB_charts.ppl.update();
  } else {
    DB_charts.ppl = new Chart(document.getElementById('db-chart-people'),{
      type:'bar',
      data:{labels,datasets:[{data,backgroundColor:'#555',borderWidth:0}]},
      options:{indexAxis:'y',responsive:true,maintainAspectRatio:false,
        plugins:{legend:{display:false}},
        scales:{x:{ticks:{stepSize:1}},y:{ticks:{font:{size:10}}}},
      }
    });
  }
}

// ─── TABLE ───────────────────────────────────────────────────────────────────
function renderTable(){
  const sortMul = DB_sort_dir==='asc'?1:-1;
  const sorted  = [...DB_filtered].sort((a,b)=>{
    const av=a[DB_sort_col]||'', bv=b[DB_sort_col]||'';
    if(typeof av==='number'||!isNaN(parseFloat(av))){
      return ((parseFloat(av)||0)-(parseFloat(bv)||0))*sortMul;
    }
    return av<bv?-sortMul:av>bv?sortMul:0;
  });

  document.querySelectorAll('#db-table th[data-col]').forEach(th=>{
    th.classList.remove('sort-asc','sort-desc');
    if(th.dataset.col===DB_sort_col) th.classList.add(DB_sort_dir==='asc'?'sort-asc':'sort-desc');
  });

  const start = (DB_page_num-1)*DB_PAGE_SIZE;
  const page  = sorted.slice(start, start+DB_PAGE_SIZE);
  const total = sorted.length;
  const pages = Math.ceil(total/DB_PAGE_SIZE);

  const tbody = document.getElementById('db-tbody');
  tbody.innerHTML='';
  page.forEach((t,i)=>{
    const n = start+i+1;
    const vMatch = t.vendor_can?'db-co-matched':'';
    const pMatch = t.purchaser_can?'db-co-matched':'';
    const tr = document.createElement('tr');
    tr.style.cursor='pointer';
    tr.innerHTML=`
      <td>${n}</td>
      <td>${t.sale_date||'—'}</td>
      <td>${t.city||'—'}</td>
      <td style="white-space:nowrap">${(t.property_class||'').slice(0,8)}</td>
      <td>${t.property_type||'—'}</td>
      <td title="${t.description||''}">${(t.description||'').slice(0,22)}</td>
      <td class="${vMatch}" title="${t.vendor_entity||''}">${(t.vendor_entity||'—').slice(0,22)}</td>
      <td class="${pMatch}" title="${t.purchaser_entity||''}">${(t.purchaser_entity||'—').slice(0,22)}</td>
      <td class="db-price">${fmtPrice(t.sale_price)}</td>
      <td class="db-price">${t.unit_price?fmtPrice(t.unit_price):'—'}</td>
      <td>${fmtArea(t.bldg_area||t.site_area, t.bldg_units||t.site_units)}</td>`;
    const detailId = `dbr-${t.txn_id}`;
    tr.onclick=()=>DB_toggleRow(detailId, t);
    tbody.appendChild(tr);

    const det = document.createElement('tr');
    det.className='db-row-detail';
    det.id=detailId;
    det.innerHTML=`<td colspan="11"><div class="db-detail-grid">
      <div class="db-detail-row"><span class="dl">Address</span><span class="dv">${t.address||'—'}</span></div>
      <div class="db-detail-row"><span class="dl">Legal Desc</span><span class="dv">${t.legal_description||'—'}</span></div>
      <div class="db-detail-row"><span class="dl">Subdivision</span><span class="dv">${t.subdivision||'—'}</span></div>
      <div class="db-detail-row"><span class="dl">Land Use</span><span class="dv">${t.land_use||'—'}</span></div>
      <div class="db-detail-row"><span class="dl">Ownership</span><span class="dv">${t.ownership_type||'—'}</span></div>
      <div class="db-detail-row"><span class="dl">Year Built</span><span class="dv">${t.year_built||'—'}</span></div>
      <div class="db-detail-row"><span class="dl">Site Area</span><span class="dv">${fmtArea(t.site_area,t.site_units)}</span></div>
      <div class="db-detail-row"><span class="dl">Bldg Area</span><span class="dv">${fmtArea(t.bldg_area,t.bldg_units)}</span></div>
      <div class="db-detail-row"><span class="dl">Vendor</span><span class="dv">${t.vendor_entity||'—'}${t.vendor_person?' · '+t.vendor_role+': '+t.vendor_person:''}${t.vendor_can?' [CORES '+t.vendor_can+']':''}</span></div>
      <div class="db-detail-row"><span class="dl">Purchaser</span><span class="dv">${t.purchaser_entity||'—'}${t.purchaser_person?' · '+t.purchaser_role+': '+t.purchaser_person:''}${t.purchaser_can?' [CORES '+t.purchaser_can+']':''}</span></div>
    </div></td>`;
    tbody.appendChild(det);
  });

  document.getElementById('db-pg-info').textContent = `Page ${DB_page_num} of ${pages} (${total.toLocaleString()} rows)`;
  document.getElementById('db-pg-prev').disabled = DB_page_num<=1;
  document.getElementById('db-pg-next').disabled = DB_page_num>=pages;
}

window.DB_toggleRow = function(id, t){
  const el = document.getElementById(id);
  if(el) el.classList.toggle('open');
};

window.DB_sort = function(col){
  if(DB_sort_col===col) DB_sort_dir=DB_sort_dir==='asc'?'desc':'asc';
  else{DB_sort_col=col;DB_sort_dir='desc';}
  DB_page_num=1;
  renderTable();
};

window.DB_page = function(dir){
  const pages = Math.ceil(DB_filtered.length/DB_PAGE_SIZE);
  DB_page_num = Math.max(1, Math.min(pages, DB_page_num+dir));
  renderTable();
};

window.DB_resetAll = function(){
  document.querySelectorAll('#db-f-class input, #db-f-type input, #db-f-own input').forEach(el=>el.checked=true);
  ['db-f-city','db-f-subdiv'].forEach(id=>document.getElementById(id).value='');
  ['db-f-year-from','db-f-year-to','db-f-price-from','db-f-price-to','db-f-company','db-f-person'].forEach(id=>document.getElementById(id).value='');
  document.getElementById('db-f-cores-only').checked=false;
  DB_applyFilters();
};

window.DB_exportCSV = function(){
  const cols = ['txn_id','sale_date','city','property_class','property_type','ownership_type',
    'description','address','vendor_entity','vendor_person','vendor_role','vendor_can',
    'purchaser_entity','purchaser_person','purchaser_role','purchaser_can',
    'sale_price','unit_price','bldg_area','site_area','year_built','subdivision'];
  const csv  = [cols.join(',')].concat(DB_filtered.map(t=>
    cols.map(c=>JSON.stringify(t[c]||'')).join(',')
  )).join('\n');
  const a = document.createElement('a');
  a.href = URL.createObjectURL(new Blob([csv],{type:'text/csv'}));
  a.download = 'cores_gettel_filtered.csv';
  a.click();
};

// ─── INIT ────────────────────────────────────────────────────────────────────
DB_applyFilters();

})();
</script>
"""

with open("dashboard_section.html","w",encoding="utf-8") as f:
    f.write(html)

print("dashboard_section.html written.")
print(f"  File size: {len(html)/1024:.0f} KB (excluding embedded data)")


In [ ]:
# @title 4 — ASSEMBLE FINAL HTML
# Inputs: mindmap_section.html, dashboard_section.html
# Output: cores_gettel_explorer.html

import re, os

with open("mindmap_section.html","r",encoding="utf-8") as f:
    mm = f.read()

with open("dashboard_section.html","r",encoding="utf-8") as f:
    db = f.read()

def extract_blocks(html, tag):
    pattern = rf'<{tag}[^>]*>(.*?)</{tag}>'
    return re.findall(pattern, html, re.DOTALL)

def strip_blocks(html, tag):
    pattern = rf'<{tag}[^>]*>.*?</{tag}>'
    return re.sub(pattern, '', html, flags=re.DOTALL).strip()

mm_styles  = extract_blocks(mm, 'style')
db_styles  = extract_blocks(db, 'style')
mm_scripts = extract_blocks(mm, 'script')
db_scripts = extract_blocks(db, 'script')
mm_body    = strip_blocks(strip_blocks(mm, 'style'), 'script').strip()
db_body    = strip_blocks(strip_blocks(db, 'style'), 'script').strip()

# Deduplicate the Google Fonts import (both sections share it)
FONT_IMPORT = "@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600&family=DM+Mono:wght@400;500&display=swap');"
all_styles = '\n'.join(mm_styles + db_styles)
all_styles = all_styles.replace(FONT_IMPORT, '', 1)  # keep only one

final = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>CORES + GETTEL Explorer</title>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/d3/7.8.5/d3.min.js"></script>
  <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
  <style>
    {FONT_IMPORT}
    *{{box-sizing:border-box;margin:0;padding:0;}}
    body{{font-family:'DM Sans',sans-serif;background:#f8f8f6;color:#1a1a1a;height:100vh;overflow:hidden;}}
    .top-nav{{display:flex;align-items:center;gap:0;background:#111;padding:0 20px;height:42px;}}
    .top-nav .nav-brand{{color:#fff;font-size:12px;font-weight:600;letter-spacing:.06em;text-transform:uppercase;margin-right:20px;}}
    .nav-tab{{padding:0 18px;height:42px;font-family:inherit;font-size:12px;font-weight:500;color:rgba(255,255,255,.6);background:none;border:none;border-bottom:2px solid transparent;cursor:pointer;letter-spacing:.02em;}}
    .nav-tab.active{{color:#fff;border-bottom-color:#fff;}}
    .nav-tab:hover{{color:#fff;}}
    .tab-content{{display:none;height:calc(100vh - 42px);overflow:hidden;}}
    .tab-content.active{{display:block;}}
    {all_styles}
  </style>
</head>
<body>

  <nav class="top-nav">
    <span class="nav-brand">CORES + GETTEL</span>
    <button class="nav-tab active" data-tab="mindmap"   onclick="switchTab('mindmap')">Network Map</button>
    <button class="nav-tab"        data-tab="dashboard" onclick="switchTab('dashboard')">Dashboard</button>
  </nav>

  <div id="tab-mindmap"   class="tab-content active">
    {mm_body}
  </div>
  <div id="tab-dashboard" class="tab-content">
    {db_body}
  </div>

  <script>
    var _mmInited = false;
    var _dbInited = false;

    function switchTab(tab) {{
      document.querySelectorAll('.nav-tab').forEach(b=>b.classList.toggle('active', b.dataset.tab===tab));
      document.querySelectorAll('.tab-content').forEach(c=>c.classList.remove('active'));
      document.getElementById('tab-'+tab).classList.add('active');
      if(tab==='mindmap' && !_mmInited) {{ _mmInited=true; initMindmap(); }}
      if(tab==='dashboard' && !_dbInited) {{ _dbInited=true; initDashboard(); }}
    }}

    // Mindmap init (deferred until tab shown)
    function initMindmap() {{
      {chr(10).join(mm_scripts)}
    }}

    // Dashboard init (deferred until tab shown)
    function initDashboard() {{
      {chr(10).join(db_scripts)}
    }}

    // Auto-init mindmap on first load
    _mmInited = true;
    initMindmap();
  </script>

</body>
</html>"""

with open("cores_gettel_explorer.html","w",encoding="utf-8") as f:
    f.write(final)

size_mb = os.path.getsize("cores_gettel_explorer.html") / 1024 / 1024
print(f"cores_gettel_explorer.html written.")
print(f"  Total size: {size_mb:.2f} MB")
print(f"  (Data size scales with your input — this measures the HTML+JS template only)")


In [ ]:
# @title 5 — DOWNLOAD FINAL HTML
from google.colab import files
files.download("cores_gettel_explorer.html")
